# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os

print(os.path.exists(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
))

True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
print("""
Method Choice:

Method:
Random Forest Classifier

Why this method fits:
- The goal is to rank pages by refresh opportunity.
- Random Forest can learn relationships between multiple search signal
Features used:
- impressions_90ds.
- It handles non-linear patterns between content age, impressions, CTR, and position.

- clicks_90d
- ctr
- avg_position
- content_age_days

The model output is used as a decision-support score for ranking content pages.
""")


Method Choice:

Method:
Random Forest Classifier

Why this method fits:
- The goal is to rank pages by refresh opportunity.
- Random Forest can learn relationships between multiple search signals.
- It handles non-linear patterns between content age, impressions, CTR, and position.

Features used:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days

The model output is used as a decision-support score for ranking content pages.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
print("""
Split Design:

A standard train/test split is used.

Reason:
The dataset contains historical content performance signals.
The split allows evaluation on unseen samples.

Design:
- 80% training data
- 20% testing data

Leakage prevention:
- No future-window features used.
- No target-derived features included.
- Only historical search signals are used.
""")



Split Design:

A standard train/test split is used.

Reason:
The dataset contains historical content performance signals.
The split allows evaluation on unseen samples.

Design:
- 80% training data
- 20% testing data

Leakage prevention:
- No future-window features used.
- No target-derived features included.
- Only historical search signals are used.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score


# Find dataset automatically
dataset_path = None

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            dataset_path = os.path.join(root, file)

print("Dataset location:")
print(dataset_path)


# Load dataset
df = pd.read_csv(dataset_path)

print("Dataset loaded")
print("Rows:", len(df))


# Create refresh opportunity target
df["refresh_target"] = (
    (df["content_age_days"] > 300)
    &
    (df["impressions_90d"] > df["impressions_90d"].median())
).astype(int)


# Features
features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]


X = df[features]
y = df["refresh_target"]


# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)


# Evaluate model
prediction = model.predict(X_test)

model_precision = precision_score(
    y_test,
    prediction
)


print("Model Precision:", model_precision)


# Create model ranking score
df["model_score"] = model.predict_proba(X)[:, 1]


# Create Week 4 baseline score
df["baseline_score"] = (
    (df["content_age_days"] / df["content_age_days"].max()) * 0.4
    +
    (df["impressions_90d"] / df["impressions_90d"].max()) * 0.3
    +
    (1 - df["ctr"] / df["ctr"].max()) * 0.3
)


# Compare top recommendations
comparison = pd.DataFrame({
    "Baseline Top Score": [
        df["baseline_score"].max()
    ],
    "Model Top Score": [
        df["model_score"].max()
    ]
})


print("\nBaseline vs Model:")
display(comparison)


print("""
Interpretation:

Baseline:
A manually created scoring rule from Week 4.

Model:
A Random Forest model learning patterns from search signals.

The comparison shows how the learned model ranks refresh opportunities
compared with the original rule-based approach.
""")

Dataset location:
/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Dataset loaded
Rows: 30000
Model Precision: 1.0

Baseline vs Model:


,Baseline Top Score,Model Top Score
0,0.980431,1.0



Interpretation:

Baseline:
A manually created scoring rule from Week 4.

Model:
A Random Forest model learning patterns from search signals.

The comparison shows how the learned model ranks refresh opportunities
compared with the original rule-based approach.



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
print("""
Error Analysis and Interpretation:

Model errors:

1. False positives:
The model may rank some pages as high refresh opportunities even when they
already perform well.

Possible reason:
Older content with strong impressions may not actually need updates.

2. False negatives:
Some pages may be missed by the model.

Possible reason:
Important content quality factors are not available in the dataset.

3. Signal limitations:
The model mainly depends on historical search signals:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days


Feature interpretation:

The model learns patterns from available search signals.
Higher scores indicate pages that are worth reviewing.

Important limitation:

The model provides decision-support and prioritization guidance.
It does not prove that refreshing a page will directly improve traffic.

Observed relationships are directional and should be validated by content teams.
""")


Error Analysis and Interpretation:

Model errors:

1. False positives:
The model may rank some pages as high refresh opportunities even when they
already perform well.

Possible reason:
Older content with strong impressions may not actually need updates.

2. False negatives:
Some pages may be missed by the model.

Possible reason:
Important content quality factors are not available in the dataset.

3. Signal limitations:
The model mainly depends on historical search signals:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days


Feature interpretation:

The model learns patterns from available search signals.
Higher scores indicate pages that are worth reviewing.

Important limitation:

The model provides decision-support and prioritization guidance.
It does not prove that refreshing a page will directly improve traffic.

Observed relationships are directional and should be validated by content teams.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.